# ARWO overlap audit
Compare the supplied workbook against the preserved CSV. See docs/arwo-expenses-extension.md for interpretation and source limitations.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
root = Path.cwd() if Path('python').exists() else Path.cwd().parent
spec = importlib.util.spec_from_file_location('arwo', root / 'python/10-extract_arwo_expenses.py')
arwo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arwo)
old = pd.read_csv(root / 'data/un-secretariat-expenses.csv')
new = pd.read_excel(root / 'data/internal/ARWO_2019-2025.xlsx', sheet_name='Data_Raw_19-25')
historical = new[new.YEAR.isin(old.YEAR.unique())]
assert arwo.records(old, arwo.FINANCIAL_FIELDS) == arwo.records(historical, arwo.FINANCIAL_FIELDS)
print(f'{len(old)} historical financial records match, including multiplicity.')
display(new.groupby(['YEAR','SOURCE_TYPE']).AMOUNT.agg(['count','sum']))
display(new[new.YEAR > 2023].groupby('YEAR').AMOUNT.agg(['count','sum']))
